In [1]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\project_repository\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [2]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=30)).date()
start = f"{start_date}T00:00:00Z"
type_names = ["Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
              "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])
list_of_owners = ['HTOC Org']
final_results = []

# Query indicators
for owner in list_of_owners:
    display(f"Querying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators?tql={tql_encoded}&fields=tags,observations,associatedGroups,falsePositives,threatAssess,&resultStart=0&resultLimit=10000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
    except Exception as e:
        display(f"Failed to query indicators for {owner}: {e}")

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].str.split().str[0].str.strip()
    observed_src.drop_duplicates(subset='indicator', inplace=True)
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start)]
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)


'Querying owner: HTOC Org'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,source,url,hostName,dnsActive,whoisActive,address,sha256,md5,lastFalsePositive,indicator
0,5629499555060684,2025-06-16T17:42:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-17T17:07:44Z,3.0,54.0,2.33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.70.100.6
1,4562574,2024-04-11T14:28:40Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-17T15:37:43Z,1.0,24.0,1.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,104.21.3.76
2,4397426,2023-08-28T13:07:38Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-17T15:37:43Z,1.0,9.0,1.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72.21.210.29
3,5629499555060705,2025-06-16T17:42:38Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-17T15:32:43Z,3.0,54.0,2.33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.42.116.175
4,5629499571090459,2025-10-08T11:23:53Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-17T14:47:43Z,1.0,30.0,2.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.195.240.19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1573,4517160,2024-02-05T17:10:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-05T23:23:34Z,3.0,67.0,3.00,...,https://realinvestmentadvice.com/,realinvestmentadvice.com/,NaN,NaN,NaN,NaN,NaN,NaN,NaN,realinvestmentadvice.com/
1574,4512619,2024-01-26T20:41:51Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-01-28T12:55:28Z,3.0,70.0,3.00,...,https://shorturl.asia,shorturl.asia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,shorturl.asia
1575,4491603,2023-12-21T16:13:01Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2023-12-22T11:56:27Z,3.0,89.0,3.00,...,https://financier.com,financier.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,financier.com
1576,4482005,2023-11-30T16:50:06Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2023-12-01T11:40:35Z,3.0,88.0,3.00,...,https://pub.marq.com/,pub.marq.com/,NaN,NaN,NaN,NaN,NaN,NaN,NaN,pub.marq.com/


In [8]:
# Load indicators from CSV file and get their records from observed_src
csv_path = r"C:\Users\jaskew\Documents\Large1088.csv"
indicators_df = pd.read_csv(csv_path)

# Determine the indicator column name in indicators_df
indicator_col = None
for col in indicators_df.columns:
    if 'indicator' in col.lower() or 'summary' in col.lower() or col.lower() in ['ioc', 'value']:
        indicator_col = col
        break

if indicator_col is None and len(indicators_df.columns) > 0:
    # Use first column if no clear indicator column found
    indicator_col = indicators_df.columns[0]

if indicator_col and 'indicator' in observed_src.columns:
    csv_indicators = indicators_df[indicator_col].dropna().unique().tolist()
    matched_records = observed_src[observed_src['indicator'].isin(csv_indicators)]
    
    display(f"Found {len(matched_records)} records from observed_src for {len(csv_indicators)} CSV indicators:")
    display(matched_records)
    
    # Show which indicators from CSV were not found
    found_indicators = set(matched_records['indicator'].unique())
    not_found = [ind for ind in csv_indicators if ind not in found_indicators]
    if not_found:
        display(f"\n{len(not_found)} indicators from CSV not found in observed_src")
else:
    display("Could not match indicators - check column names")
    display(f"indicators_df columns: {list(indicators_df.columns)}")
    display(f"observed_src columns: {list(observed_src.columns) if not observed_src.empty else 'Empty DataFrame'}")

'Found 931 records from observed_src for 931 CSV indicators:'

'Found 931 records from observed_src for 931 CSV indicators:'

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,source,url,hostName,dnsActive,whoisActive,address,sha256,md5,lastFalsePositive,indicator
0,5629499555060684,2025-06-16T17:42:22Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-17T17:07:44Z,3.0,54.0,2.33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,109.70.100.6
1,4562574,2024-04-11T14:28:40Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-17T15:37:43Z,1.0,24.0,1.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,104.21.3.76
2,4397426,2023-08-28T13:07:38Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-17T15:37:43Z,1.0,9.0,1.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72.21.210.29
3,5629499555060705,2025-06-16T17:42:38Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-17T15:32:43Z,3.0,54.0,2.33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,192.42.116.175
4,5629499571090459,2025-10-08T11:23:53Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-12-17T14:47:43Z,1.0,30.0,2.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,91.195.240.19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1480,5629499542289506,2025-05-29T19:31:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-12-04T13:37:43Z,0.0,0.0,0.00,...,NaN,NaN,biologyinsights.com,False,False,NaN,NaN,NaN,NaN,biologyinsights.com
1483,4245566,2022-08-10T14:19:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,File,2025-12-03T21:32:43Z,3.0,100.0,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,AE8293EA164A55787E48E4373C875EC8C2AD2AF47C22A9...,NaN,NaN,AE8293EA164A55787E48E4373C875EC8C2AD2AF47C22A9...
1502,5629499556005846,2025-06-30T12:21:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-08T13:14:30Z,1.0,0.0,1.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.49.1.127
1506,5629499556005840,2025-06-30T12:21:12Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-10-26T22:41:47Z,1.0,0.0,0.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.62.156.59


In [10]:
# Extract and count all tags from matched_records
tags_col = None
for col in ['tags.data', 'tags']:
    if col in matched_records.columns:
        tags_col = col
        break

if tags_col is None:
    display("No tags field found in matched_records")
else:
    all_tags = []
    
    for _, row in matched_records.iterrows():
        tags_val = row[tags_col]
        
        # Normalize to list of tag dicts/strings
        if isinstance(tags_val, dict) and "data" in tags_val:
            tags = tags_val.get("data", [])
        elif isinstance(tags_val, list):
            tags = tags_val
        else:
            tags = []
        
        for tag in tags:
            if isinstance(tag, dict):
                tag_name = tag.get("name") or tag.get("tag") or str(tag)
            else:
                tag_name = str(tag)
            
            if tag_name and tag_name != 'nan':
                all_tags.append(tag_name)
    
    if all_tags:
        tag_counts = pd.Series(all_tags).value_counts()
        display(f"Most common tags (Total unique tags: {len(tag_counts)}):")
        display(tag_counts.head(50))
        
        # Create a summary dataframe
        tag_summary = pd.DataFrame({
            'tag': tag_counts.index,
            'count': tag_counts.values
        }).reset_index(drop=True)
        display("\nTop 50 tags as DataFrame:")
        display(tag_summary.head(50))
    else:
        display("No tags found in matched_records")

'Most common tags (Total unique tags: 333):'

Observed                         764
VA CSOC CTS Splunk               753
malicious                        695
CMS Splunk API                   672
OS Splunk API                    663
FDA Splunk API                   643
HHS Splunk API                   628
HRSA Splunk API                  624
DHA Splunk API                   607
IHS Splunk API                   606
NIH Splunk API                   586
CDC Splunk API                   421
VA OIS CSOC CTS Splunk           113
suspicious                        64
I&W                               61
T-Suite                           44
Iran                              43
UNC5537                           42
INDICATOR NOTICE 25178.1          40
Mr Hamza Group                    40
DDoS                              40
Hacktivism                        40
External Remote Services          25
hrsa_wl                           24
Russia                            18
CMS                               18
UNC3944                           17
s

'\nTop 50 tags as DataFrame:'

,tag,count
0,Observed,764
1,VA CSOC CTS Splunk,753
2,malicious,695
3,CMS Splunk API,672
4,OS Splunk API,663
5,FDA Splunk API,643
6,HHS Splunk API,628
7,HRSA Splunk API,624
8,DHA Splunk API,607
9,IHS Splunk API,606


In [11]:
# Break down associatedGroups.data
group_col = None
for col in ['associatedGroups.data', 'associatedGroups']:
    if col in matched_records.columns:
        group_col = col
        break

if group_col is None:
    display("No associated groups field found in matched_records")
else:
    group_rows = []
    
    for _, row in matched_records.iterrows():
        indicator = row.get('indicator')
        groups_val = row[group_col]
        
        # Normalize to list of group dicts
        if isinstance(groups_val, dict) and "data" in groups_val:
            groups = groups_val.get("data", [])
        elif isinstance(groups_val, list):
            groups = groups_val
        else:
            groups = []
        
        for grp in groups:
            if isinstance(grp, dict):
                group_rows.append({
                    'indicator': indicator,
                    'group_type': grp.get('type') or grp.get('typeName') or grp.get('groupType'),
                    'group_name': grp.get('name') or grp.get('summary') or grp.get('title'),
                    'owner': grp.get('ownerName'),
                    'id': grp.get('id')
                })
    
    if group_rows:
        groups_df = pd.DataFrame(group_rows)
        
        display(f"Total associated group entries: {len(groups_df)}")
        display(f"Unique groups: {groups_df['group_name'].nunique()}")
        display(f"\nAll associated groups:")
        display(groups_df)
        
        # Group type breakdown
        display(f"\n{'='*80}")
        display("Group Type Breakdown:")
        type_counts = groups_df['group_type'].value_counts()
        display(type_counts)
        
        # Most common groups by type
        display(f"\n{'='*80}")
        display("Most Common Groups by Type:")
        for group_type in type_counts.index[:5]:  # Top 5 types
            mask = groups_df['group_type'] == group_type
            top_groups = groups_df[mask]['group_name'].value_counts().head(10)
            display(f"\n{group_type} (Top 10):")
            display(top_groups)
        
        # Overall most common groups
        display(f"\n{'='*80}")
        display("Overall Most Common Groups (Top 30):")
        group_name_counts = groups_df['group_name'].value_counts().head(30)
        display(group_name_counts)
        
    else:
        display("No associated groups found in matched_records")

'Total associated group entries: 372'

'Unique groups: 138'

'\nAll associated groups:'

,indicator,group_type,group_name,owner,id
0,104.21.3.76,Campaign,H-ISAC Credential Phishing Indicator Sharing |...,HTOC Org,330116
1,72.21.210.29,Campaign,CISA 'Oral B' themed Phishing Email Observed i...,HTOC Org,160812
2,91.195.240.19,Campaign,Mapping the Infrastructure and Malware Ecosyst...,HTOC Org,6755399465000876
3,91.195.240.19,Report,German IP associated with MuddyWater Activity ...,HTOC Org,5629499558001272
4,91.195.240.19,Adversary,G0069/MuddyWater/ATK51/Boggy Serpens/COBALT UL...,HTOC Org,442870
...,...,...,...,...,...
367,link.edgepilot.com,Campaign,"(Target Org), eSign Review, Remittance Ajustme...",HTOC Org,5629499566000137
368,biologyinsights.com,Incident,NIH IRT - 67180 MDE Suspicious command in RunM...,HTOC Org,6755399450169960
369,AE8293EA164A55787E48E4373C875EC8C2AD2AF47C22A9...,Incident,VA Potential Phishing [CSETS:000013684 – 20220...,HTOC Org,140089
370,23.205.105.180,Incident,NIH IRT - 65503 MicrosoftDefenderForEndpoint -...,HTOC Org,6755399448004157


'\n================================================================================'

...

'\nDocument (Top 10):'

group_name
JAR 16-20296A GRIZZLY STEPPE 2016-1229                                    10
CMS Anomali ThreatStream Indicators from 01.14.2024 through 02.13.2024     2
LookingGlass Collection COVID-19 IOCs                                      2
LookingGlass Collection COVID-19 IOCs (TLP: Amber)                         1
CMS Anomali ThreatStream Indicators 02.14.2024 through 03.14.2024          1
16-00014614: Overview of Tsar Team Espionage Activity                      1
VA-NSOC SAR S-17-034 Copier Themed Phishing Campaign Targets VA Users      1
Name: count, dtype: int64

'\n================================================================================'

'Overall Most Common Groups (Top 30):'

group_name
UNC5537                                                                                                                                                                                                42
INDICATOR NOTICE 25178.1 – Iran-Aligned Hacktivist Groups Launch DDoS Attacks Against U.S. Networks                                                                                                    40
G1015/UNC3944/Scattered Spider/Scatteredspider/DEV-0671/DEV-0971/DEV0875/LUCR-3/Muddled Libra/Octo Tempest/Roasted 0ktapus/Scatter Swine/Starfraud/Storm-0875                                          17
CMS CCIC CTI Intelligence Brief 09.18.2023                                                                                                                                                             16
UNC5537 Data Theft and Extortion Targeting Snowflake Data Instances                                                                                                                  